<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/buildmemoryagent(w3d1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#given question:
#Build a simple memory + tool invocation agent (e.g., fetch + summarize web content)

In [ ]:
#open ai api key
import os
openai_api_key = input("Please enter your OpenAI API key (e.g., sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx): ")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API key configured successfully.")

In [3]:
pip install -q langchain==0.1.16 langchain_openai==0.1.3 beautifulsoup4 html2text requests tiktoken

In [10]:
import requests
from bs4 import BeautifulSoup
from html2text import html2text
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# Tool 1: Fetch web content
@tool
def fetch_web_content(url: str) -> str:
    """Fetches content from a web URL and returns it as clean, plain text."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for script_or_style in soup(['script', 'style', 'header', 'footer', 'nav']):
            script_or_style.extract()
        clean_text = html2text(soup.get_text())
        return clean_text.strip()
    except requests.exceptions.RequestException as e:
        return f"Error fetching content from {url}: {e}"

# Tool 2: Summarize text
@tool
def summarize_text(text: str) -> str:
    """Summarizes the given text using an AI model. Input should be a string of text."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = f"Please summarize the following text concisely:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content

In [11]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# List the tools the agent can use
tools = [fetch_web_content, summarize_text]
# Defining how the agent should think and respond (a simple prompt template)
agent_prompt_template = """You are a helpful assistant. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Previous conversation history:
{chat_history}
Question: {input}
Thought:{agent_scratchpad}"""

# Create the prompt for the agent
agent_prompt = PromptTemplate.from_template(agent_prompt_template)

# Set up memory for the agent to remember past interactions
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    k=3
)
# Create the agent itself using the model, tools, and prompt
agent = create_react_agent(llm, tools, agent_prompt)
# Create the AgentExecutor to run the agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, memory=memory, handle_parsing_errors=True)

In [9]:
#invoking the agent
print("Agent is ready. Let's try an example!")
response = agent_executor.invoke({
    "input": "Fetch the content from https://developers.llamaindex.ai/python/framework/understanding/rag/ and summarize it in one sentence."
})
print("\n--- Agent's Final Output ---")
print(response["output"])

Agent is ready. Let's try an example!


> Entering new AgentExecutor chain...
I need to fetch the content from the specified URL and then summarize it in one sentence.  
Action: fetch_web_content  
Action Input: https://developers.llamaindex.ai/python/framework/understanding/rag/  Introduction to RAG | Developer Documentation Skip to content LlamaIndex FrameworkLearnBuilding a RAG pipelineIntroduction to RAGCopy MarkdownOpen in ClaudeOpen in ChatGPTOpen in CursorCopy MarkdownView as MarkdownIntroduction to RAG Tip If you haven’t, install LlamaIndex and complete the starter tutorial before you read this. It will help ground these steps in your experience. LLMs are trained on enormous bodies of data but they aren’t trained on your data. Retrieval-Augmented Generation (RAG) solves this problem by adding your data to the data LLMs already have access to. You will see references to RAG frequently in this documentation. Query engines, chat engines and agents often use RAG to complete their t